## Problem 1: Visualise data in an interactive map (10 points)


Create an *interactive* **thematic map** using the skills you learnt during lesson 5. You can use *Folium*,<br>
or any other suitable Python package - feel free to experiment).

The map should contain more than one layer (from at least two different data sets). Pay attention to <br>
classification and visualisation (appropriate colour scheme choice, etc.). Write your code in this <br>
notebook, and store the resulting maps in `.html` format (plus possibly additional files, depending on<br>
your approach) in the `NOTEBOOK_DIRECTORY / "docs"` folder.

### Topic of the map

- Use any of the data sets we have used during the course (travel time data, population grid), or <br>
  any other data set of your interest (for example, [open data sets from Statistics Finland](https://www.stat.fi/org/avoindata/index_en.html), <br>
  or the open geo-data from [the Helsinki Region Infoshare](https://hri.fi/en)). If you download <br>
  data sets, add, commit, and push them to the `DATA_DIRECTORY`. Be sure to include an attribution <br>
  to each data set in your output map!
- Feel free to start from and adapt examples from the course material! You can take one step further <br>
  and add additional analysis to, e.g., the shopping centre dominance or the population within vincinity <br>
  of certain addresses)
- This should **not** be an interactive version of your submission for *problem 1*! Create something new!

### Criteria

- The map has more than one data layer (for instance, the results of an analysis, and the road network <br>
  to illustrate a complex spatial pattern). If (and only if) you think it adds additional value, feel <br>
  free to add a base map (but a base map does not count as an own layer) (2 points)
- The map portrays the output of an analysis or a classification that adds value beyond just displaying <br>
  the raw data (2 points)
- The map follows good cartographic practices (appropriate map extent and zoom level, colour scheme, <br>
  legend and data sources, etc.) (3 points)
- The map demonstrates the skills learnt during lesson 5, such as interactive elements (popups, etc.). (3 points)

### Output

- Save the map(s) in html format in the `NOTEBOOK_DIRECTORY / "docs"` folder, remember to add, commit, <br>
  and push them to the GitHub repository
- Remember to also add, commit, and push input data sets and code

---

In [1]:
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"

In [2]:
# ADD YOUR OWN CODE HERE

In [3]:
import geopandas
# Load roads in Turku to geodataframe
roads = geopandas.read_file(
    DATA_DIRECTORY
    / "turku_roads.gpkg"
)
# Load schools in Turku to geodataframe
schools = geopandas.read_file(
    DATA_DIRECTORY
    / "turku_schools_accidents.gpkg"
)
schools.head()

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/cartogis2026/share/proj failed


,gml_id,id,oltyp_nimi,til_vuosi,tunn,onimi,olo,oltyp,euref_x,euref_y,accident_count,geometry
0,oppilaitokset.80,80,Peruskoulut,2025,08548,Aunelan koulu,0,11,236163.0,6711922.0,1,POINT (236163 6711922)
1,oppilaitokset.112,112,Peruskoulut,2025,08808,Cygnaeus skola,0,11,239598.0,6711255.0,13,POINT (239598 6711255)
2,oppilaitokset.212,212,Peruskoulut,2025,03760,Haarlan koulu,0,11,236611.0,6704848.0,0,POINT (236611 6704848)
3,oppilaitokset.247,247,Peruskoulut,2025,08549,Hannunniitun koulu,0,11,242972.0,6711487.0,0,POINT (242972 6711487)
4,oppilaitokset.304,304,Peruskoulut,2025,08550,Hepokullan koulu,0,11,238854.0,6713900.0,0,POINT (238854 6713900)


In [4]:
# Calculate how many meters of roads with a speed limit of 50 km/h or higher are within 500 meters of each school

# First, create a 500m buffer for each school
schools['buffer'] = schools.geometry.buffer(distance=500)
# For joining based on the buffer, we need to set the buffer as the geometry
schools_buffers = schools.set_geometry("buffer")

# Select roads with speed limit >= 50 km/h
fast_roads = roads[
    roads["arvo"] >= 50].copy()

# Next, intersect the roads with school buffers. Using overlay so that just the length of road that is 
# inside the buffer gets counted in, not the whole road
roads_in_buffers = geopandas.overlay(
    schools_buffers,
    fast_roads,
    how="intersection",
    keep_geom_type=False

)
# Calculate length of road segments inside each buffer
roads_in_buffers["road_length"] = roads_in_buffers.geometry.length
# Sum road lengths for each school
road_lengths = round(roads_in_buffers.groupby("onimi")["road_length"].sum())

# Add result to original schools GeoDataFrame
schools["fast_roads_length"] = (schools["onimi"].map(road_lengths).fillna(0)
)
schools.head()


,gml_id,id,oltyp_nimi,til_vuosi,tunn,onimi,olo,oltyp,euref_x,euref_y,accident_count,geometry,buffer,fast_roads_length
0,oppilaitokset.80,80,Peruskoulut,2025,08548,Aunelan koulu,0,11,236163.0,6711922.0,1,POINT (236163 6711922),"POLYGON ((236663 6711922, 236660.592 6711872.9...",3500.0
1,oppilaitokset.112,112,Peruskoulut,2025,08808,Cygnaeus skola,0,11,239598.0,6711255.0,13,POINT (239598 6711255),"POLYGON ((240098 6711255, 240095.592 6711205.9...",0.0
2,oppilaitokset.212,212,Peruskoulut,2025,03760,Haarlan koulu,0,11,236611.0,6704848.0,0,POINT (236611 6704848),"POLYGON ((237111 6704848, 237108.592 6704798.9...",1695.0
3,oppilaitokset.247,247,Peruskoulut,2025,08549,Hannunniitun koulu,0,11,242972.0,6711487.0,0,POINT (242972 6711487),"POLYGON ((243472 6711487, 243469.592 6711437.9...",651.0
4,oppilaitokset.304,304,Peruskoulut,2025,08550,Hepokullan koulu,0,11,238854.0,6713900.0,0,POINT (238854 6713900),"POLYGON ((239354 6713900, 239351.592 6713850.9...",2755.0


In [28]:
# Keeping only necessary columns for schools and school buffers
schools = schools[["oltyp_nimi", "onimi", "geometry", "fast_roads_length"]]
schools_buffers = schools_buffers[["oltyp_nimi", "onimi", "buffer"]]
# Save the layers to data folder
schools.to_file(DATA_DIRECTORY / "schools_fast_roads_length.gpkg")
schools_buffers.to_file(DATA_DIRECTORY / "schools_buffers_fast_roads_length.gpkg")

In [6]:
# Creating a classified column of the length of 50km/h + roads around schools
# This will be used for visualization
import mapclassify

# Define class breaks
class_breaks = [0, 1000, 2000, 3000]

# Create classifier
classifier = mapclassify.UserDefined.make(
    bins=class_breaks
)

# Create a new classified column
schools["road_length_class"] = classifier(schools["fast_roads_length"]
)
# Assign colors for each class
colors = {
    0: "#2b83ba",  # Blue: 0 m
    1: "#ffffbf",  # Light yellow: >0–1000 m
    2: "#fdae61",  # Orange: >1000–2000 m
    3: "#f46d43",  # Dark orange: >2000–3000 m
    4: "#d7191c"   # Red: >3000 m
}

In [27]:
import folium
import branca
from IPython.display import display, IFrame

interactive_map = folium.Map(
    location=(60.45, 22.26),
    zoom_start=11,
    control_scale=True
)
# Creating a popup that gives information about the school when the marker is clicked
popup = folium.GeoJsonPopup(
    fields=["onimi", "oltyp_nimi", "fast_roads_length"],
    aliases=["School name", "Type of school", "Length of roads >= 50km/h within 500m:"],
    localize=True,
    labels=True,
    style="background-color: yellow;",
)
# Adding the 500m buffers around the schools
schools_buffers_layer = folium.features.GeoJson(
    schools_buffers,
    name="500 m buffer around schools",
    color="blue",
    opacity = 0.5,
    weight=1,
    fill_color="blue",
    fill_opacity=0.1,
    fill=True,
)
schools_buffers_layer.add_to(interactive_map)

roads_in_buffers_layer = folium.features.GeoJson(
    roads_in_buffers,
    name="Roads with over 50 km/h speed limit close to schools",
    color = "red",
    weight = 1.5,
    opacity=0.8
).add_to(interactive_map)

schools_layer = folium.GeoJson(
    schools,
    name="Schools",
    marker=folium.CircleMarker(
        radius=6,
        fill = True
    ),
    # Style the markers based on the classification
    style_function=lambda feature: {
        "fillColor": colors[feature["properties"]["road_length_class"]],
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.9,
        "fill": True
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["onimi", "fast_roads_length"],
        aliases=[
            "School:",
            "Roads ≥50 km/h within 500 m (m):"
    ]),
    popup=popup,
    
)
schools_layer.add_to(interactive_map)

# Create the legend
legend_html = '''
{% macro html(this, kwargs) %}
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 260px; height: 200px; 
     border:2px solid grey; z-index:9999; font-size:13px;
     background-color:white; opacity: 0.9;">

<b>Legend</b><br>
<b>Length of roads ≥50 km/h within 500 m of schools<br>

     <i class="fa fa-circle" style="color:#2b83ba"></i>
     &nbsp; 0 m<br>
     
     <i class="fa fa-circle" style="color:#ffffbf"></i>
     &nbsp; 0–1000 m<br>

     <i class="fa fa-circle" style="color:#fdae61"></i>
     &nbsp; 1000–2000 m<br>

     <i class="fa fa-circle" style="color:#f46d43"></i>
     &nbsp; 2000–3000 m<br>

     <i class="fa fa-circle" style="color:#d7191c"></i>
     &nbsp; >3000 m<br>

     <hr style="margin: 6px 0;">

     <span style="color:red; font-weight:bold;">━━</span>
     &nbsp; Road with speed limit ≥50 km/h<br>

     <i class="fa fa-circle" style="color:#98CEF8"></i>
     &nbsp; 500 m school buffer<br>
    
</div>
{% endmacro %}
'''

legend = branca.element.MacroElement()
legend._template = branca.element.Template(legend_html)

# Add the legend to the map
interactive_map.get_root().add_child(legend)


# Add option to toggle on or off layers
folium.LayerControl().add_to(interactive_map)
interactive_map


# Save map
interactive_map.save(OUTPUT_DIRECTORY / "turku_schools.html")